In [11]:
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from typing import List, Literal
from enum import Enum
import json
from pathlib import Path
from tqdm.asyncio import tqdm_asyncio

client = AsyncOpenAI()

In [12]:
class ProceedSignal(str, Enum):
    """Investment decision signal."""
    STRONG_YES = "strong_yes"
    QUESTIONABLE = "questionable"
    STRONG_NO = "strong_no"


class DealVerdict(BaseModel):
    """Structured verdict on a potential PE deal."""
    
    proceed_signal: ProceedSignal = Field(
        description="Overall investment signal: strong_yes (high conviction opportunity), questionable (needs more analysis), or strong_no (pass on deal)"
    )
    
    evidence_pro: List[str] = Field(
        description="Key positive factors supporting investment: competitive advantages, growth drivers, financial strengths, market opportunities",
        default_factory=list
    )
    
    evidence_contra: List[str] = Field(
        description="Key concerns or risks against investment: competitive threats, financial weaknesses, market headwinds, execution risks",
        default_factory=list
    )
    
    similar_deals: str = Field(
        description="Comparable past PE deals (fictional examples) in similar sectors/situations that provide context. Format as brief descriptions of buyouts, growth equity, add-on acquisitions, etc."
    )
    
    key_information_requested: str = Field(
        description="Critical additional information, analysis, or due diligence needed to make final investment decision, even if signal is positive"
    )


class TrainingExample(BaseModel):
    """Complete training data example for fine-tuning."""
    
    report: str = Field(
        description="The generated fake company research report"
    )
    
    reasoning: str = Field(
        description="Step-by-step analytical reasoning that led to the verdict, covering competition, customers, financials, and growth"
    )
    
    verdict: DealVerdict = Field(
        description="The structured investment verdict with proceed signal and supporting evidence"
    )

In [13]:
async def generate_training_example(
    example_report_path: str,
    model: str = "gpt-4.1-mini",
    temperature: float = 2.0,
    client: AsyncOpenAI = None,
) -> TrainingExample:
    """
    Generate synthetic training data by creating a FAKE company report and corresponding verdict.
    
    The example report is used as a TEMPLATE for structure/format only.
    All data (company name, numbers, facts) in the generated report will be fictional.
    """
    
    # Read the example report as a template
    example_report_text = Path(example_report_path).read_text()
    
    system_prompt = """You are a synthetic training data generator for private equity deal analysis.

Your task is to generate COMPLETELY FICTIONAL training data consisting of:
1. A fake company research report (similar structure to the example provided)
2. An investment verdict on that fake company
3. Analytical reasoning about that fake verdict, how to come to that.

The example report provided is ONLY a template for structure and format. You must:

- Invent a completely new company (different industry, different name, different business model)
- Generate realistic but fake financial data (revenue, margins, growth rates, balance sheet)
- Create fake competitors, customers, and market dynamics
- Make all facts, figures, and dates fictional but internally consistent

Then, analyze YOUR OWN generated fake report through these lenses:
1. COMPETITION: Market position, competitive dynamics, defensibility
2. CUSTOMERS: Customer quality, retention, concentration risks  
3. FINANCIALS: Growth trajectory, profitability, cash generation, balance sheet
4. GROWTH OPPORTUNITIES: Organic growth, M&A, operational improvements

Finally, determine:
- proceed_signal: strong_yes (compelling opportunity), questionable (uncertain/needs work), strong_no (pass)
- evidence_pro: specific positive factors from YOUR fake report (3-6 points)
- evidence_contra: specific concerns from YOUR fake report (3-6 points)
- similar_deals: fictional PE deals that provide comps (e.g., "Thoma Bravo's $2.1B buyout of ConnectWise in 2019")
- key_information_requested: critical additional diligence needed

Generate diverse, realistic example. Vary industries, company stages, financial profiles, and verdict outcomes."""

    user_prompt = f"""Here is an EXAMPLE report structure (DO NOT analyze this company - it's just a template):

{example_report_text}

---

Now generate a COMPLETELY NEW fictional company and deal memo in the same format, then provide your reasoning and investment verdict for YOUR generated company."""
    
    response = await client.responses.parse(
        model=model,
        temperature=temperature,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        text_format=TrainingExample,
    )
    
    return response.output_parsed

In [14]:
async def generate_batch(
    example_report_path: str,
    num_examples: int = 10,
    client: AsyncOpenAI = None,
    temperature: float = 1.5,
) -> List[TrainingExample]:
    """
    Generate multiple DIVERSE training examples using the same report as a template.
    
    Each example will be a completely different fake company with unique:
    - Company name and industry
    - Financial data
    - Competitive landscape
    - Investment verdict
    
    Args:
        example_report_path: Path to example report (used as structure template only)
        num_examples: Number of diverse fake companies to generate
        client: AsyncOpenAI client instance
        temperature: Sampling temperature (high = more diversity)
        
    Returns:
        List of TrainingExample objects, each with a unique fake company
    """
    import asyncio
    
    tasks = [
        generate_training_example(
            example_report_path=example_report_path,
            temperature=temperature,
            client=client
        )
        for _ in range(num_examples)
    ]
    
    results = await tqdm_asyncio.gather(*tasks, desc="Generating training examples")
    return results

In [15]:
def save_training_data(
    examples: List[TrainingExample],
    output_path: str = "training_data.jsonl"
):
    """
    Save training examples to JSONL format for fine-tuning.
    
    Args:
        examples: List of TrainingExample objects
        output_path: Path to output JSONL file
    """
    with open(output_path, 'w') as f:
        for example in examples:
            # Convert to dict and write as JSON line
            f.write(example.model_dump_json() + '\n')
    
    print(f"Saved {len(examples)} training examples to {output_path}")

In [ ]:
batch = await generate_batch(
    example_report_path="../data/output/Duolingo_2025-11-18_21-47-42.md",
    num_examples=2,
    client=client,
    temperature=2.0  # Maximum creativity for diverse fake companies
)

Generating training examples:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Save to file
save_training_data(batch, "../data/training_data_examples.jsonl")